In [1]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [2]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_mensajeria = config['MENSAJERIA_OLTP']
    config_etl = config['ETL_PROCESS']

# Construct the database URL
url_mensajeria = (f"{config_mensajeria['drivername']}://{config_mensajeria['user']}:{config_mensajeria['password']}@{config_mensajeria['host']}:"
          f"{config_mensajeria['port']}/{config_mensajeria['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_mensajeria)
etl_conn = create_engine(url_etl)

In [ ]:
dim_sede = pd.read_sql_table('sede',mensajeria)
dim_sede.head()


<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   sede_id          52 non-null     int64
 1   nombre           52 non-null     str  
 2   direccion        52 non-null     str  
 3   telefono         52 non-null     str  
 4   nombre_contacto  52 non-null     str  
 5   ciudad_id        52 non-null     int64
 6   cliente_id       52 non-null     int64
dtypes: int64(3), str(4)
memory usage: 3.0 KB


In [ ]:
dim_sede.drop(columns=['nombre_contacto','direccion','telefono','ciudad_id'],inplace=True)

In [ ]:
dim_sede[dim_sede.duplicated(subset='nombre')]

In [ ]:
dim_sede.drop_duplicates(inplace=True)

In [ ]:
dim_sede.to_sql('dim_sede', etl_conn, if_exists='replace',index_label='key_dim_sede')